# IZQAR IEEE-CIS v4 — causal frequency features, GPU CV

Research mode only (3 expanding chronological folds inside the first 85% of data).
The 15% chronological tail is never evaluated. Two identical-fold arms run:
`freq=causal` (v4 candidate) vs `freq=none` (v3 behavior).
No production artifact is exported.

In [ ]:
from pathlib import Path
import json, subprocess, sys

EXPECTED_REVISION = '95175417157b509b35cca2bcd0e43b4950015d93'
root = Path('/kaggle/working/izqar_ieee_v4')
if root.exists():
    raise FileExistsError('Use fresh session')
subprocess.run(['git', 'clone', '--branch', 'feat/ieee-cis-research', 'https://github.com/Night011/izqar.git', str(root)], check=True)
subprocess.run(['git', 'checkout', '--detach', EXPECTED_REVISION], cwd=root, check=True)
actual = subprocess.check_output(['git', 'rev-parse', 'HEAD'], cwd=root, text=True).strip()
assert actual == EXPECTED_REVISION, f'unexpected revision {actual}'
backend = root / 'backend'
subprocess.run([sys.executable, '-m', 'pip', 'install', '--no-cache-dir', '-q', '-r', str(backend / 'requirements-ml.txt')], check=True)
matches = list(Path('/kaggle/input').rglob('train_transaction.csv'))
assert len(matches) == 1, f'expected one train_transaction.csv, got {matches}'
data_dir = matches[0].parent
assert (data_dir / 'train_identity.csv').is_file(), 'train_identity.csv missing'
print('input:', data_dir, 'bytes:', matches[0].stat().st_size, 'revision:', actual)
import xgboost; print('xgboost', xgboost.__version__)

In [ ]:
import subprocess, sys, json
from pathlib import Path

results = {}
for arm in ['causal', 'none']:
    out = Path(f'/kaggle/working/ieee_v4_{arm}.joblib')
    subprocess.run([sys.executable, str(backend / 'scripts' / 'train_ieee_v3.py'),
                    '--data-dir', str(data_dir), '--output', str(out),
                    '--mode', 'research', '--freq', arm, '--device', 'cuda',
                    '--estimators', '1500'], cwd=backend, check=True)
    report_path = out.with_suffix('.cv.metrics.json')
    results[arm] = json.loads(report_path.read_text(encoding='utf-8'))
    print(f'== {arm} ==', json.dumps(results[arm]['mean_metrics'], indent=2))

summary = {
    'expected_revision': EXPECTED_REVISION,
    'actual_revision': actual,
    'mode': 'research',
    'arms': {arm: {'mean_metrics': results[arm]['mean_metrics'],
                   'folds': [{'valid_frac': f['valid_frac'],
                              'pr_auc': f['metrics']['average_precision_pr_auc'],
                              'f1': f['metrics']['f1']} for f in results[arm]['folds']]}
             for arm in results},
    'test_evaluation_performed': False,
}
Path('/kaggle/working/ieee_v4_comparison.json').write_text(json.dumps(summary, indent=2), encoding='utf-8')
print('written /kaggle/working/ieee_v4_comparison.json')